In [128]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [129]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import RidgeCV

In [130]:
df = pd.read_csv(r"C:\Users\ayesh\Downloads\house-prices-advanced-regression-techniques\train.csv")

In [131]:
df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [132]:
missing = df.isnull().sum().sort_values(ascending=False)

In [133]:
missing[missing > 0]

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

In [134]:
df.shape

(1460, 81)

In [135]:
df.drop(columns=['PoolQC', 'MSSubClass'], inplace=True)

In [136]:
cols_none = [
    'MiscFeature','Alley','Fence','FireplaceQu',
    'GarageType','GarageFinish','GarageQual','GarageCond',
    'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
    'MasVnrType'
]

for col in cols_none:
    df[col] = df[col].fillna("None")

In [137]:
df.select_dtypes(exclude = ['int64', 'float64']).columns

Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'Fence', 'MiscFeature', 'SaleType',
       'SaleCondition'],
      dtype='str')

In [138]:
cols_zero = [
    'GarageYrBlt','MasVnrArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF',
    'TotalBsmtSF','BsmtFullBath','BsmtHalfBath',
    'GarageCars','GarageArea'
]

for col in cols_zero:
    df[col] = df[col].fillna(0)


In [139]:
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage']\
                     .transform(lambda x: x.fillna(x.median()))

In [140]:
cols_mode = [
    'MSZoning','Utilities','Functional',
    'Exterior1st','Exterior2nd','KitchenQual','SaleType'
]

for col in cols_mode:
    df[col] = df[col].fillna(df[col].mode()[0])

In [141]:
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

In [142]:
df.isnull().sum()[df.isnull().sum() > 0]

Series([], dtype: int64)

In [143]:
quality_features = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']
qual_map = {'None': 0, 'NA': 0, 'Ex': 5, 'Gd': 4,'TA': 3, 'Fa': 2,'Po': 1}
for cols in quality_features:
    df[cols] = df[cols].map(qual_map)

In [144]:
bsmt_exp_map = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
df['BsmtExposure'] = df['BsmtExposure'].map(bsmt_exp_map)

In [145]:
bsmt_fin_map = {'None':0, 'Unf':1, 'LwQ':2, 'Rec':3, 'BLQ':4, 'ALQ':5, 'GLQ':6}
df['BsmtFinType1'] = df['BsmtFinType1'].map(bsmt_fin_map)
df['BsmtFinType2'] = df['BsmtFinType2'].map(bsmt_fin_map)

In [146]:
functional_map = {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7}
df['Functional'] = df['Functional'].map(functional_map)
garage_finish_map = {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
df['GarageFinish'] = df['GarageFinish'].map(garage_finish_map)
paved_map = {'N': 0, 'P': 1, 'Y': 2}
df['PavedDrive'] = df['PavedDrive'].map(paved_map)
landslope_map = {'Sev': 0, 'Mod': 1,'Gtl': 2}
df['LandSlope'] = df['LandSlope'].map(landslope_map)
lotshape_map = {'IR3': 0, 'IR2': 1, 'IR1': 2,'Reg': 3, }
df['LotShape'] = df['LotShape'].map(lotshape_map)

In [147]:
categorical_cols = [
    'MSZoning', 'Street', 'Alley', 'LandContour', 'Utilities', 'LotConfig', 'Neighborhood', 'Condition1', 
    'Condition2', 'BldgType',
    'HouseStyle', 'RoofStyle', 'RoofMatl', 
    'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 'SaleType',
    'SaleCondition', 'Fence', 'MiscFeature'
]

In [148]:
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [149]:
df.dtypes

Id                    int64
LotFrontage         float64
LotArea               int64
LotShape              int64
LandSlope             int64
                     ...   
Fence_None             bool
MiscFeature_None       bool
MiscFeature_Othr       bool
MiscFeature_Shed       bool
MiscFeature_TenC       bool
Length: 205, dtype: object

In [150]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [151]:
df.dtypes.value_counts()

int64      202
float64      3
Name: count, dtype: int64

In [152]:
df.shape

(1460, 205)

In [153]:
df.drop(columns = 'Id', inplace = True)

In [154]:
y = np.log1p(df['SalePrice'])
X = df.drop('SalePrice', axis=1)

In [155]:
cols_to_drop = [
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
    '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
    'YearBuilt', 'YearRemodAdd', 'YrSold',
    'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath'
]

X = X.drop(columns=cols_to_drop)

In [156]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

In [157]:
scaler = StandardScaler()

In [158]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [159]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

In [160]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame()
vif["feature"] = X_train_scaled_df.columns
vif["VIF"] = [
    variance_inflation_factor(X_train_scaled_df.values, i)
    for i in range(X_train_scaled_df.shape[1])
]

vif = vif.sort_values(by="VIF", ascending=False)
print(vif.head(15))

c:\Users\ayesh\OneDrive\Documents\ml_new\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1784: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.uncentered_tss
c:\Users\ayesh\OneDrive\Documents\ml_new\venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


                   feature          VIF
133     Exterior2nd_CBlock          inf
119     Exterior1st_CBlock          inf
24             GarageYrBlt  1870.591469
168        GarageType_None  1721.029868
186       MiscFeature_None   795.889213
188       MiscFeature_Shed   651.035468
104        RoofStyle_Gable   251.913673
106          RoofStyle_Hip   232.759719
174           SaleType_New   123.290442
181  SaleCondition_Partial   122.127606
127    Exterior1st_VinylSd   113.123612
142    Exterior2nd_VinylSd    95.077135
41             MSZoning_RL    78.514024
123    Exterior1st_MetalSd    74.554568
163      GarageType_Attchd    68.782876


In [161]:
model = RidgeCV(alphas=[0.1, 1, 10, 50, 100], cv=5)
model.fit(X_train_scaled_df, y_train)

,"alphas alphas: array-like of shape (n_alphas,), default=(0.1, 1.0, 10.0)Array of alpha values to try.Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.If using Leave-One-Out cross-validation, alphas must be strictly positive.","[0.1, 1, ...]"
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"scoring scoring: str, callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: negative :ref:`mean squared error ` if cv is None (i.e. when using leave-one-out cross-validation), or :ref:`coefficient of determination ` (:math:`R^2`) otherwise.",None
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the efficient Leave-One-Out cross-validation- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used, else,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.",5
,"gcv_mode gcv_mode: {'auto', 'svd', 'eigen'}, default='auto'Flag indicating which strategy to use when performingLeave-One-Out Cross-Validation. Options are:: 'auto' : use 'svd' if n_samples > n_features, otherwise use 'eigen' 'svd' : force use of singular value decomposition of X when X is dense, eigenvalue decomposition of X^T.X when X is sparse. 'eigen' : force computation via eigendecomposition of X.X^TThe 'auto' mode is the default and is intended to pick the cheaperoption of the two depending on the shape of the training data.",None
,"store_cv_results store_cv_results: bool, default=FalseFlag indicating if the cross-validation values corresponding toeach alpha should be stored in the ``cv_results_`` attribute (seebelow). This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionchanged:: 1.5 Parameter name changed from `store_cv_values` to `store_cv_results`.",False
,"alpha_per_target alpha_per_target: bool, default=FalseFlag indicating whether to optimize the alpha value (picked from the`alphas` parameter list) for each target separately (for multi-outputsettings: multiple prediction targets). When set to `True`, afterfitting, the `alpha_` attribute will contain a value for each target.When set to `False`, a single alpha is used for all targets... versionadded:: 0.24",False


In [162]:
y_pred_train = model.predict(X_train_scaled_df)
y_pred_test = model.predict(X_test_scaled_df)

In [163]:
print("Train R2:", r2_score(y_train, y_pred_train))
print("Test R2:", r2_score(y_test, y_pred_test))


y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_test)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
print("RMSE:", rmse)

Train R2: 0.9166354777458338
Test R2: 0.8918559481276385
RMSE: 25016.577950068942


In [164]:
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=5)
lasso.fit(X_train_scaled_df, y_train)

,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"n_alphas n_alphas: int, default=100Number of alphas along the regularization path... deprecated:: 1.7 `n_alphas` was deprecated in 1.7 and will be removed in 1.9. Use `alphas` instead.",'deprecated'
,"alphas alphas: array-like or int, default=NoneValues of alphas to test along the regularization path.If int, `alphas` values are generated automatically.If array-like, list of alpha values to use... versionchanged:: 1.7 `alphas` accepts an integer value which removes the need to pass `n_alphas`... deprecated:: 1.7 `alphas=None` was deprecated in 1.7 and will be removed in 1.9, at which point the default value will be set to 100.",'warn'
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: 'auto', bool or array-like of shape (n_features, n_features), default='auto'Whether to use a precomputed Gram matrix to speed upcalculations. If set to ``'auto'`` let us decide. The Grammatrix can also be passed as argument.",'auto'
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``.",0.0001
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"cv cv: int, cross-validation generator or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- int, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For int/None inputs, :class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: bool or int, default=FalseAmount of verbosity.",False
,"n_jobs n_jobs: int, default=NoneNumber of CPUs to use during the cross validation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [165]:
y_pred_train = lasso.predict(X_train_scaled_df)
y_pred_test = lasso.predict(X_test_scaled_df)

In [166]:
print("Train R2:", r2_score(y_train, y_pred_train))
print("Test R2:", r2_score(y_test, y_pred_test))


y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred_test)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
print("RMSE:", rmse)

Train R2: 0.896865168022265
Test R2: 0.8900480266369237
RMSE: 25829.790195356272


In [167]:
(lasso.coef_ != 0).sum()

np.int64(67)